# Analisis Distribution Shift: Sintetik (CIC, UNS) vs Real-Traffic (AWS)

Notebook ini membandingkan **karakteristik distribusi 9 fitur SFM** antara dataset sintetik
(CSE-CIC-IDS2018 = **CIC**, UNSW-NB15 = **UNS**) dan trafik **nyata dari AWS** (hasil capture
Fase 1/2). Tujuannya membuktikan secara kuantitatif & visual bahwa trafik nyata berbeda
distribusi dari benchmark sintetik --- akar *distribution shift* yang membuat model gagal
mendeteksi serangan nyata.

Mengikuti pendekatan Layeghy dkk. (jarak Wasserstein) + teknik pelengkap (KDE/ECDF, boxplot,
PCA/t-SNE, domain classifier / proxy A-distance).

> **Data:** butuh CSV fitur AWS (`*_flows.csv` dari `unsw-far/results/`). Untuk CIC/UNS,
> notebook memakai CSV fitur bila tersedia; bila tidak, dibuat *sampel sintetik* dari statistik
> ringkas (mean/scale scaler + median attack) HANYA untuk ilustrasi bentuk grafik (ditandai jelas).
> Untuk hasil final, sediakan CSV fitur CIC/UNS asli.

## 0. Setup

In [ ]:
import importlib, sys, subprocess
need = [m for m in ('matplotlib','pandas','numpy','scipy','sklearn') if importlib.util.find_spec(m.replace('sklearn','sklearn')) is None]
pkgmap = {'sklearn':'scikit-learn'}
if need:
    subprocess.run([sys.executable,'-m','pip','install','-q',*[pkgmap.get(m,m) for m in need]], check=True)
print('setup ok' if not need else f'installed: {need}')

In [ ]:
import os, json, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10, 'axes.grid': True, 'grid.alpha': 0.3})

CANON = ['duration','fwd_pkts','bwd_pkts','fwd_bytes','bwd_bytes','fwd_mean','bwd_mean','src_load','dst_load']
CAND = ['..', '.', '../unswnb-15', 'unswnb-15']
DATA_DIR = next((d for d in CAND if os.path.exists(os.path.join(d,'cross_dataset_baseline.json'))), '..')
print('DATA_DIR =', os.path.abspath(DATA_DIR))

def load_json(name, fb=None):
    p = os.path.join(DATA_DIR, name)
    return json.load(open(p)) if os.path.exists(p) else fb

## 1. Muat data 9 fitur: CIC, UNS, AWS

Prioritas: cari CSV fitur asli. Lokasi yang dicoba (silakan sesuaikan/letakkan filenya):
- **AWS**: `results/*_flows.csv` (dari S3 `unsw-far/results/`), atau `/opt/unsw/results/`.
- **CIC/UNS**: `cic_flows.csv` / `uns_flows.csv` (9 fitur) bila kamu ekspor dari pipeline latih.

Jika CIC/UNS tak ada, dibuat sampel sintetik dari `scaler_mean/scale` (ilustratif).

In [ ]:
def find_first(paths):
    for p in paths:
        hits = sorted(glob.glob(p))
        if hits: return hits
    return []

# ---- AWS: gabung semua *_flows.csv hasil capture (D1/D2/detect) ----
aws_globs = [os.path.join(DATA_DIR,'results','*_flows.csv'), '/opt/unsw/results/*_flows.csv',
             os.path.join(DATA_DIR,'aws','results','*_flows.csv'), '*_flows.csv']
aws_files = find_first(aws_globs)
print('AWS files:', [os.path.basename(f) for f in aws_files] or 'TIDAK ADA')

def read_feats(files):
    dfs = []
    for f in files:
        try:
            d = pd.read_csv(f)
            cols = [c for c in CANON if c in d.columns]
            if len(cols)==len(CANON): dfs.append(d[CANON])
        except Exception as e:
            print('  skip', f, e)
    return pd.concat(dfs, ignore_index=True) if dfs else None

aws = read_feats(aws_files)
print('AWS flows:', 0 if aws is None else len(aws))

In [ ]:
# ---- CIC & UNS: cari CSV 9-fitur; jika tak ada -> sampel sintetik dari scaler (ILUSTRATIF) ----
cic_files = find_first([os.path.join(DATA_DIR,'cic_flows.csv'), os.path.join(DATA_DIR,'*cic*flows*.csv')])
uns_files = find_first([os.path.join(DATA_DIR,'uns_flows.csv'), os.path.join(DATA_DIR,'*uns*flows*.csv')])
cic = read_feats(cic_files); uns = read_feats(uns_files)

SYNTHETIC = False
if cic is None or uns is None:
    SYNTHETIC = True
    meta = load_json('model_efficiency.json')  # placeholder; scaler ada di deploy_meta (di S3/models)
    dm = load_json('deploy_meta_9feat.json')
    if dm is None:
        # fallback statistik kasar (mean training CIC-fit) dari catatan proyek
        mean = np.array([1.2175e7, 24.43, 6.21, 990.47, 4609.18, 50.32, 113.18, 255117.10, 15260.12])
        scale = np.array([2.0e7, 60, 20, 5000, 20000, 60, 150, 6.0e5, 4.0e4])
    else:
        mean = np.array(dm['scaler_mean']); scale = np.array(dm['scaler_scale'])
    rng = np.random.default_rng(42)
    n = len(aws) if aws is not None else 3000
    # CIC ~ N(mean, scale) dilipat non-negatif; UNS digeser (ilustrasi distribusi beda)
    cic = pd.DataFrame(np.abs(rng.normal(mean, np.abs(scale), size=(n,9))), columns=CANON)
    uns = pd.DataFrame(np.abs(rng.normal(mean*np.array([1e-6,1,2,0.4,0.2,1.4,0.5,3.5,0.1]),
                                        np.abs(scale)*0.8, size=(n,9))), columns=CANON)
    print('CIC/UNS tidak ditemukan -> pakai SAMPEL SINTETIK (ilustratif) dari scaler.')
else:
    print('CIC flows:', len(cic), '| UNS flows:', len(uns))

# Susun dict domain (buang yang None)
domains = {k:v for k,v in [('CIC',cic),('UNS',uns),('AWS-real',aws)] if v is not None and len(v)>0}
print('Domain tersedia:', list(domains.keys()), '| SYNTHETIC CIC/UNS =', SYNTHETIC)

## 2. Statistik ringkas per fitur (median antar-domain)

In [ ]:
med = pd.DataFrame({k: v[CANON].median() for k,v in domains.items()})
print('Median tiap fitur per domain:')
display(med.style.format('{:.2f}') if hasattr(med,'style') else med.round(2))
print('Perhatikan fitur laju (src_load, dst_load) & duration -> biasanya paling berbeda utk AWS.')

## 3. Jarak Wasserstein (W1) antar-domain per fitur

Dihitung di ruang z-score (di-fit pada gabungan) agar antar-fitur sebanding. Nilai besar =
distribusi makin berbeda. Mengikuti pendekatan Layeghy dkk.

In [ ]:
from scipy.stats import wasserstein_distance
from itertools import combinations

# z-score global per fitur (fit pd gabungan semua domain) supaya skala sebanding
allX = pd.concat([v[CANON] for v in domains.values()], ignore_index=True)
mu, sd = allX.mean(), allX.std().replace(0,1)
Z = {k: (v[CANON]-mu)/sd for k,v in domains.items()}

pairs = list(combinations(domains.keys(), 2))
rows = []
for a,b in pairs:
    r = {'pasangan': f'{a} vs {b}'}
    for c in CANON:
        r[c] = wasserstein_distance(Z[a][c].values, Z[b][c].values)
    r['RATA2'] = np.mean([r[c] for c in CANON])
    rows.append(r)
W = pd.DataFrame(rows).set_index('pasangan')
print('Jarak Wasserstein (z-space) per fitur + rata-rata:')
display(W.round(3))

fig, ax = plt.subplots(figsize=(7.2,3.6))
W['RATA2'].plot(kind='barh', ax=ax, color='#4C72B0')
ax.set_xlabel('W1 rata-rata (z-space)'); ax.set_title('Jarak distribusi antar-domain (makin besar = makin beda)')
for i,(idx,val) in enumerate(W['RATA2'].items()): ax.text(val, i, f' {val:.2f}', va='center')
plt.tight_layout(); plt.show()
print('Harapan: pasangan yang melibatkan AWS-real punya W1 lebih besar (domain gap).')

In [ ]:
# Heatmap W1 per fitur x pasangan
fig, ax = plt.subplots(figsize=(9,2.6+0.4*len(pairs)))
M = W[CANON].values
im = ax.imshow(M, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(CANON))); ax.set_xticklabels(CANON, rotation=45, ha='right')
ax.set_yticks(range(len(W.index))); ax.set_yticklabels(W.index)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        ax.text(j,i,f'{M[i,j]:.1f}',ha='center',va='center',fontsize=8)
fig.colorbar(im, ax=ax, label='W1'); ax.set_title('W1 per fitur x pasangan domain')
plt.tight_layout(); plt.show()

## 4. Overlay distribusi (ECDF) untuk fitur kunci

ECDF (kurva kumulatif) paling jelas menunjukkan pergeseran distribusi. Fitur kunci:
`duration`, `src_load`, `dst_load` (yang di Fase 2 terbukti paling meleset untuk AWS).

In [ ]:
def ecdf(x):
    x = np.sort(np.asarray(x, float)); y = np.arange(1,len(x)+1)/len(x); return x,y

key_feats = ['duration','src_load','dst_load','fwd_pkts']
colors = {'CIC':'#4C72B0','UNS':'#DD8452','AWS-real':'#55A868'}
fig, axes = plt.subplots(2,2, figsize=(11,7))
for ax, feat in zip(axes.ravel(), key_feats):
    for k,v in domains.items():
        vals = v[feat].replace([np.inf,-np.inf], np.nan).dropna()
        vals = vals[vals>=0]
        if len(vals)==0: continue
        xs,ys = ecdf(np.log1p(vals))  # log1p agar skala lebar terbaca
        ax.plot(xs, ys, label=k, color=colors.get(k), lw=2)
    ax.set_title(f'ECDF {feat} (log1p)'); ax.set_xlabel('log1p(nilai)'); ax.set_ylabel('proporsi <= x')
    ax.legend()
plt.suptitle('ECDF fitur kunci: kurva terpisah = distribusi berbeda', fontsize=12)
plt.tight_layout(); plt.show()

## 5. Boxplot berdampingan (semua fitur, log-scale)

In [ ]:
fig, axes = plt.subplots(3,3, figsize=(12,9))
for ax, feat in zip(axes.ravel(), CANON):
    data = []
    labels = []
    for k,v in domains.items():
        vals = v[feat].replace([np.inf,-np.inf], np.nan).dropna()
        data.append(np.log1p(vals[vals>=0].values)); labels.append(k)
    ax.boxplot(data, labels=labels, showfliers=False)
    ax.set_title(f'{feat} (log1p)')
plt.suptitle('Boxplot per fitur antar-domain (log1p, tanpa outlier)', fontsize=12)
plt.tight_layout(); plt.show()

## 6. Proyeksi 2D: PCA & t-SNE

Reduksi 9 fitur -> 2D, warnai per domain. Bila 'awan titik' terpisah -> bukti visual domain gap.
Data di-sub-sampel & di-z-score dulu. t-SNE bisa lambat -> dibatasi ~1500 titik/domain.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

CAP = 1500
parts, labs = [], []
rng = np.random.default_rng(0)
for k,v in domains.items():
    d = v[CANON].replace([np.inf,-np.inf], np.nan).dropna()
    if len(d)>CAP: d = d.iloc[rng.choice(len(d), CAP, replace=False)]
    parts.append(d.values); labs += [k]*len(d)
X = np.vstack(parts); labs = np.array(labs)
# log1p utk fitur lebar lalu z-score
Xl = np.log1p(np.clip(X, 0, None))
Xs = StandardScaler().fit_transform(Xl)

pca = PCA(n_components=2, random_state=0).fit_transform(Xs)
fig, ax = plt.subplots(figsize=(6.6,5))
for k in domains:
    m = labs==k; ax.scatter(pca[m,0], pca[m,1], s=8, alpha=0.4, label=k, color=colors.get(k))
ax.set_title('PCA 2D (9 fitur, log1p+z-score)'); ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.manifold import TSNE
try:
    ts = TSNE(n_components=2, perplexity=30, init='pca', random_state=0).fit_transform(Xs)
    fig, ax = plt.subplots(figsize=(6.6,5))
    for k in domains:
        m = labs==k; ax.scatter(ts[m,0], ts[m,1], s=8, alpha=0.4, label=k, color=colors.get(k))
    ax.set_title('t-SNE 2D (9 fitur)'); ax.legend()
    plt.tight_layout(); plt.show()
except Exception as e:
    print('t-SNE gagal/lambat:', e)

## 7. Domain classifier (proxy A-distance)

Latih classifier membedakan pasangan domain. **Akurasi (AUC) tinggi -> domain mudah dibedakan
= gap besar**; ~0.5 -> mirip. Proxy A-distance = 2(1 - 2*err).

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

def domain_gap(a, b, n=1500):
    A = domains[a][CANON].replace([np.inf,-np.inf],np.nan).dropna()
    B = domains[b][CANON].replace([np.inf,-np.inf],np.nan).dropna()
    A = A.sample(min(n,len(A)), random_state=0); B = B.sample(min(n,len(B)), random_state=0)
    X = np.log1p(np.clip(np.vstack([A.values,B.values]),0,None))
    y = np.r_[np.zeros(len(A)), np.ones(len(B))]
    clf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=0, n_jobs=-1)
    acc = cross_val_score(clf, X, y, cv=5, scoring='accuracy').mean()
    a_dist = 2*(1 - 2*(1-acc))  # proxy A-distance
    return acc, max(0.0, a_dist)

res = []
for a,b in pairs:
    acc, ad = domain_gap(a,b)
    res.append({'pasangan': f'{a} vs {b}', 'akurasi_pembeda': round(acc,3), 'proxy_A_distance': round(ad,3)})
dg = pd.DataFrame(res).set_index('pasangan')
display(dg)
print('Akurasi ~0.5 = domain mirip; mendekati 1.0 = domain sangat berbeda (gap besar).')

## 8. Rangkuman

- **Wasserstein** & **domain classifier** memberi ukuran kuantitatif jarak antar-domain.
- **ECDF/boxplot/PCA/t-SNE** memberi bukti visual.
- **Harapan temuan:** trafik **AWS-real** terpisah jelas dari CIC & UNS, terutama pada fitur
  *laju* (`src_load`,`dst_load`) & `duration` -> inilah *distribution shift* yang menjelaskan
  mengapa model benchmark gagal mendeteksi serangan nyata (Fase 2), dan mengapa **kalibrasi
  domain** diperlukan.

> **Untuk hasil final (bukan ilustratif):** sediakan CSV 9-fitur asli CIC & UNS (mis. diekspor
> dari pipeline latih di SageMaker) sebagai `cic_flows.csv` / `uns_flows.csv` di folder data,
> lalu jalankan ulang. Bila `SYNTHETIC=True`, grafik CIC/UNS hanya ilustrasi bentuk.